# CSE 150A: HMM for Developer Seniority Prediction

**Team:** Iha Gadiya, Ioanna Gkerdouki, Lianna Lim, Ved Panse, William Diego

**Goal:** Model how developer seniority (a latent variable) influences observable survey responses using a Hidden Markov Model.

## PEAS Analysis

- **Performance measure:** Log-likelihood of observation sequences, accuracy of decoded hidden states compared to experience-based ground truth labels.
- **Environment:** Professional developers from the United States who work at companies with 1,000 to 4,999 employees.
- **Actuators:** The agent outputs a probability distribution over seniority levels (junior, mid-level, senior) and decodes the most likely hidden state sequence.
- **Sensors:** `AISelect`, `YearsCode`, `YearsCodePro`, `CodingActivities`, `DevType`, `OrgSize`, and `ConvertedCompYearly`.

The problem we are solving is: given a sequence of observable developer attributes, can we infer their underlying seniority level over time? The HMM allows us to model seniority as a hidden state that transitions (career progression) and emits observable features (survey responses).

## Dataset Overview

We used the Stack Overflow Developer Survey 2023 dataset from Kaggle. Our cleaned dataset is in `cleaned_data/cleaned_stackoverflow_bn_data.csv`.

**Filtering criteria:**
- Country: United States of America
- OrgSize: 1,000 to 4,999 employees
- DevType: Contains "Developer"
- No missing values in selected columns

**Final dataset:** 1,106 rows

### Variables Used

| Variable | Values | Description |
| --- | --- | --- |
| `DevType` | full_stack, back_end, front_end, mobile, applications, embedded, qa, advocate, game_graphics, experience | Developer role |
| `AISelect` | yes, no, plan_to | AI tool usage |
| `YearsCode` | low, medium, high | Total coding experience |
| `YearsCodePro` | low, medium, high | Professional coding experience |
| `CodingActivities` | codes_as_hobby, no_code_as_hobby | Codes outside work |
| `ConvertedCompYearly` | low, medium, high, very_high | Yearly compensation quartile |

## Latent Variable Identification (Ioanna)

The latent variable in our dataset is **Developer Seniority**, which represents a developer's career stage or experience level, such as junior, mid-level, or senior. This variable is latent because the Stack Overflow survey does not directly ask respondents to classify themselves into explicit seniority tiers. Instead, we observe features that are influenced by a developer's underlying seniority level. A developer's seniority affects the types of roles they hold (DevType), their likelihood of adopting AI tools (AISelect), whether they code outside of work (CodingActivities), and their yearly compensation (ConvertedCompYearly). For example, a senior developer is more likely to hold a back-end or full-stack position, earn a higher salary, and have established coding habits compared to a junior developer. In our Bayesian Network, we used ConvertedCompYearly as a proxy for skill, with observed features pointing toward salary as the outcome. However, in the HMM, we reverse this perspective: Developer Seniority becomes the hidden state that generates the observed survey responses. The hidden states (junior, mid-level, senior) transition over time to model career progression, and each state probabilistically emits the observed features. This structure aligns with the HMM framework, where the hidden state causally influences the observations rather than being the end result of them.

## Part 2: Temporal Data Construction (Iha)

This dataset does not contain an explicit timestamp or natural temporal ordering between samples. However, the variables YearsCodePro and YearsCode model career progression and can be used to convert tabular data into a sequential format. 

Developer Seniority represents a developer's career stage or experience level, such as junior, mid-level, or senior. This latent variable is dependent on time and thus can be categorized by years of coding experience using the variables YearsCodePro and YearsCode. 

To impose sequential structure on the tabular dataset, we first sort the samples by YearsCodePro (low, medium, high). This variable serves as the primary temporal ordering because professional coding experience most naturally reflects progression through a software development career. Within each YearsCodePro category, samples are ordered again using YearsCode, representing total coding experience. This acts as a tie breaker between developers with similar professional experience levels.

The observed variables used as HMM emissions are DevType, AISelect, CodingActivities, and ConvertedCompYearly. These variables are discretized prior to HMM training to allow the observations to be represented as finite emission symbols.

In [9]:
import os
import glob
import shutil
import pandas as pd

data_path = "../data/survey_results_public.csv"
df = pd.read_csv(data_path)

# Only keep the wanted columns listed below:
cols = [
    "Country",
    "OrgSize",
    "DevType",
    "AISelect",
    "YearsCode",
    "YearsCodePro",
    "CodingActivities",
    "ConvertedCompYearly"
]
df = df[cols].copy()
# Print dimensions
print(f"Shape: {df.shape}")

Shape: (89184, 8)


In [10]:
# Filter data to ensure each field only has a certain value
# Country: United States of America
# Org: 500 - 999 employees
# Role: Developer

filtered = df[
    (df["Country"] == "United States of America") &
    (df["OrgSize"] == ("1,000 to 4,999 employees")) &
    (df["DevType"].str.contains("Developer"))
].copy()

# Show shape post filtering
print(f"Shape post filtering: {filtered.shape}")

Shape post filtering: (1285, 8)


In [11]:
# Remove rows with missing values
filtered = filtered.dropna().copy()

# Show shpae after removing
print(f"Shape post drop missing values: {filtered.shape}")

Shape post drop missing values: (1106, 8)


In [12]:
# Convert strings into numerical 

# Helper function to convert string to number
def convert_years(x):
    if x == "More than 50 years":
        return 51
    elif x == "Less than 1 year":
        return 0.5
    else:
        return float(x)
filtered["YearsCode_num"] = filtered["YearsCode"].apply(convert_years)
filtered["YearsCodePro_num"] = filtered["YearsCodePro"].apply(convert_years)

In [13]:
# Discretize number of years coding into three categories

# Helper function to discretize
def discretize_years(years):
    if years < 3:
        return "low"
    elif years < 10:
        return "medium"
    else:
        return "high"
    
# Descritize both the pro and the non pro coding years
filtered["YearsCode_cat"] = filtered["YearsCode_num"].apply(discretize_years)
filtered["YearsCodePro_cat"] = filtered["YearsCodePro_num"].apply(discretize_years)

# Show counts in each category
print(filtered["YearsCode_cat"].value_counts())
print()
print(filtered["YearsCodePro_cat"].value_counts())

YearsCode_cat
high      830
medium    274
low         2
Name: count, dtype: int64

YearsCodePro_cat
high      596
medium    422
low        88
Name: count, dtype: int64


In [14]:
# Simplify salary into 4 buckets: low, medium, high, very high based on quartiles
def discretize_salary(data):
    data = data.copy()
    
    q1 = data['ConvertedCompYearly'].quantile(0.25)
    q2 = data['ConvertedCompYearly'].quantile(0.5)
    q3 = data['ConvertedCompYearly'].quantile(0.75)

    salary_labels = []
    
    for val in data['ConvertedCompYearly']:
        if val < q1:
            salary_labels.append("low")
        elif val >= q1 and val < q2:
            salary_labels.append("medium")
        elif val >= q2 and val < q3:
            salary_labels.append("high")
        else:
            salary_labels.append("very_high")

    data = data.copy()
    data["ConvertedCompYearly_cat"] = salary_labels
    return data

filtered = discretize_salary(filtered)
filtered.head()

,Country,OrgSize,DevType,AISelect,YearsCode,YearsCodePro,CodingActivities,ConvertedCompYearly,YearsCode_num,YearsCodePro_num,YearsCode_cat,YearsCodePro_cat,ConvertedCompYearly_cat
6,United States of America,"1,000 to 4,999 employees","Developer, full-stack",Yes,4,3,Hobby;Contribute to open-source projects;Profe...,135000.0,4.0,3.0,medium,medium,medium
42,United States of America,"1,000 to 4,999 employees","Developer, front-end",Yes,21,16,Hobby;Freelance/contract work,165000.0,21.0,16.0,high,high,high
273,United States of America,"1,000 to 4,999 employees","Developer, full-stack","No, and I don't plan to",11,7,I don’t code outside of work,240000.0,11.0,7.0,high,medium,very_high
536,United States of America,"1,000 to 4,999 employees","Developer, mobile","No, and I don't plan to",16,14,Hobby;Contribute to open-source projects,140000.0,16.0,14.0,high,high,medium
959,United States of America,"1,000 to 4,999 employees","Developer, back-end","No, and I don't plan to",30,28,Hobby;Professional development or self-paced l...,160000.0,30.0,28.0,high,high,high


In [15]:
# Simplify the devloper titles into easier to see categories
def simplify_type_of_dev(dev):
    dev = str(dev)
    
    if "Developer, full-stack" in dev:
        return "full_stack"
    elif "Developer, back-end" in dev:
        return "back_end"
    elif "Developer, front-end" in dev:
        return "front_end"
    elif "Developer, mobile" in dev:
        return "mobile"
    elif "Developer, desktop or enterprise applications" in dev:
        return "applications"
    elif "Developer, embedded applications or devices" in dev:
        return "embedded"
    elif "Developer, QA or test" in dev:
        return "qa"
    elif "Experience" in dev:
        return "experience"
    elif "Developer, game or graphics" in dev:
        return "game_graphics"
    elif "Developer Advocate" in dev:
        return "advocate"
    else:
        return "other_developer"
filtered["DevType_cat"] = filtered["DevType"].apply(simplify_type_of_dev)

print(filtered["DevType_cat"].value_counts())

DevType_cat
full_stack       512
back_end         304
applications      80
front_end         77
embedded          44
mobile            38
qa                19
experience        14
game_graphics     11
advocate           7
Name: count, dtype: int64


In [16]:
# Simplify list of hobbies into hobby vs no hobby
def simplify_hobbies(hobby):
    hobby = str(hobby)
    if "Hobby" in hobby:
        return "codes_as_hobby"
    else:
        return "no_code_as_hobby"
filtered["CodingActivities_cat"] = filtered["CodingActivities"].apply(simplify_hobbies)
print(filtered["CodingActivities_cat"].value_counts())

CodingActivities_cat
codes_as_hobby      784
no_code_as_hobby    322
Name: count, dtype: int64


In [17]:
# Simplify the AI field for easier use 
def simplify_AI(ai):
    ai = str(ai)
    if "No" and "but" in ai:
        return "plan_to"
    elif "No" in ai:
        return "no"
    else:
        return "yes"
filtered["AISelect_cat"] = filtered["AISelect"].apply(simplify_AI)
print(filtered["AISelect_cat"].value_counts())

AISelect_cat
no         422
yes        388
plan_to    296
Name: count, dtype: int64


In [18]:
# Simplify the org field for readability 
def simplify_org(ai):
    return "medium_size"
filtered["OrgSize_cat"] = filtered["OrgSize"].apply(simplify_org)
print(filtered["OrgSize_cat"].value_counts())

OrgSize_cat
medium_size    1106
Name: count, dtype: int64


In [19]:
# Simplify the country field for readability 
def simplify_country(ai):
    return "USA"
filtered["Country_cat"] = filtered["Country"].apply(simplify_country)
print(filtered["Country_cat"].value_counts())

Country_cat
USA    1106
Name: count, dtype: int64


In [24]:
# Remove the extra columns created when discretizing the data and set the correct
# names to the columns
bn_data = pd.DataFrame({
    "DevType": filtered["DevType_cat"],
    "AISelect": filtered["AISelect_cat"],
    "YearsCode": filtered["YearsCode_cat"],
    "YearsCodePro": filtered["YearsCodePro_cat"],
    "CodingActivities": filtered["CodingActivities_cat"],
    "ConvertedCompYearly": filtered["ConvertedCompYearly_cat"],
})
print(f"Final Shape: {bn_data.shape}")
bn_data.head()

Final Shape: (1106, 6)


,DevType,AISelect,YearsCode,YearsCodePro,CodingActivities,ConvertedCompYearly
6,full_stack,yes,medium,medium,codes_as_hobby,medium
42,front_end,yes,high,high,codes_as_hobby,high
273,full_stack,no,high,medium,no_code_as_hobby,very_high
536,mobile,no,high,high,codes_as_hobby,medium
959,back_end,no,high,high,codes_as_hobby,high


In [25]:
#sorting by year

order = {
    "low": 0,
    "medium": 1,
    "high": 2
}

bn_data = bn_data.sort_values(
    by=["YearsCodePro", "YearsCode"],
    key=lambda col: col.map(order)
)
bn_data.head()

,DevType,AISelect,YearsCode,YearsCodePro,CodingActivities,ConvertedCompYearly
49475,qa,no,low,low,codes_as_hobby,low
74134,full_stack,yes,low,low,no_code_as_hobby,low
5961,front_end,yes,medium,low,codes_as_hobby,medium
6101,back_end,no,medium,low,no_code_as_hobby,medium
6158,full_stack,yes,medium,low,codes_as_hobby,low


In [26]:
# Save the final data into a csv
bn_data.to_csv("../cleaned_data/cleaned_stackoverflow_hmm_data.csv", index=False)

## Part 3: HMM Implementation and Parameter Estimation (Lianna)

*Implementation of a basic discrete HMM with EM-based parameter estimation.*

**TODO:**
- Initial state distribution (π)
- Transition matrix (A)
- Emission matrix (B)
- EM-based learning

## Part 4: Inference (Will)

*Implementation of forward-backward algorithm for computing observation-sequence likelihood and decoded hidden state sequences.*

In [12]:
import numpy as np
import pandas as pd

In [13]:
# input, for now set to empty
transition_matrix = []
emission_matrix = []
prior = []
evidence = []

In [14]:
class HMM:
    """A Hidden markov model which takes Transition matrix and Sensor matrix as inputs"""

    def __init__(self, transition_matrix, emission_matrix, prior):
        self.transition_matrix = np.array(transition_matrix, dtype=float)
        self.emission_matrix = np.array(emission_matrix, dtype=float)
        self.prior = np.array(prior, dtype=float)

    def emission_dist(self, ev):
        return self.emission_matrix[:, ev]

In [15]:
def normalize(vector):
    sum = np.sum(vector)
    return vector / sum

In [16]:
# Forward
def forward_step(HMM, fv, ev):
    prediction = HMM.transition_matrix.T @ fv
    weighted = HMM.emission_dist(ev) * prediction
    return normalize(weighted)

# Backward
def backward_step(HMM, b, ev):
    weighted = HMM.emission_dist(ev) * b
    beta = HMM.transition_matrix @ weighted
    return normalize(beta)

In [17]:
def forward_backward(HMM, evidence):
    t = len(evidence)
    num_states = len(HMM.prior)
    
    evidence = list(evidence)
    evidence.insert(0, None)  # makes evidence[1] correspond to time 1

    fv = [np.zeros(num_states) for _ in range(t + 1)]
    b = np.ones(num_states)
    graph_array = [np.zeros(num_states) for _ in range(t)]

    fv[0] = HMM.prior

    print('Begin forward algorithm')

    for i in range(1, t + 1):
        # compute the forward message at time i
        fv[i] = forward_step(HMM, fv[i - 1], evidence[i])

        print(f'Forward @ {i} is {fv[i]}')

    print()
    print('Begin backward algorithm and smoothing')

    for i in range(t, 0, -1):
        # then normalize to get the smoothed posterior at time i
        graph_array[i - 1] = normalize(fv[i] * b)

        # update the backward message
        b = backward_step(HMM, b, evidence[i])

        print(f'Backward @ {i} is {b}')

    solution = np.argmax(np.array(graph_array), axis=1)
    return (graph_array, solution)

In [18]:
# checks if there is data, prevents a crash before prior parts completed
if len(evidence) > 0:
    salary_HMM = HMM(transition_matrix, emission_matrix, prior)
    graph_array, solution = forward_backward(salary_HMM, evidence)
else:
    print("No Data Yet")

No Data Yet


## Part 5: Evaluation and Reflection

*Evaluate results and compare HMM to Bayesian Network.*

**TODO:**
- Compare decoded states to ground truth (if available)
- Use log-likelihood or another metric
- Reflection (1-2 paragraphs): What does the HMM capture that the BN does not? What assumptions does the HMM impose that the BN does not require?